In [1]:
!nvidia-smi

Wed Jun 18 07:43:52 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls -lh "/content/drive/MyDrive/YoloV8/Dataset.zip"


-rw------- 1 root root 619M Jun 18 07:26 /content/drive/MyDrive/YoloV8/Dataset.zip


In [4]:
import zipfile

zip_path = "/content/drive/MyDrive/YoloV8/Dataset.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [5]:
import shutil
import os

# Path to the outer 'Dataset' folder (the one that wraps another 'Dataset')
outer_dataset_path = "/content/Dataset/Dataset"
inner_dataset_path = "/content/Dataset"

# Move contents up one level
for item in os.listdir(outer_dataset_path):
    src = os.path.join(outer_dataset_path, item)
    dst = os.path.join(inner_dataset_path, item)
    shutil.move(src, dst)

# Remove the now-empty folder
shutil.rmtree(outer_dataset_path)

print("✅ Removed extra 'Dataset' folder. Structure is now clean.")


✅ Removed extra 'Dataset' folder. Structure is now clean.


In [6]:
import os
from PIL import Image

# Updated class names (no 'Person' or 'Man')
class_names = ['Human body', 'Missile', 'Rifle', 'Tank', 'Helicopter']
class_to_id = {name: idx for idx, name in enumerate(class_names)}

# Dataset folders
base_path = '/content/Dataset'
splits = ['train', 'test', 'validation']

for split in splits:
    split_path = os.path.join(base_path, split)
    for class_name in os.listdir(split_path):
        class_dir = os.path.join(split_path, class_name)
        if not os.path.isdir(class_dir): continue

        if class_name not in class_to_id:
            print(f"❌ Skipping unlisted class: {class_name}")
            continue

        label_dir = os.path.join(class_dir, 'labels')
        os.makedirs(label_dir, exist_ok=True)

        for fname in os.listdir(class_dir):
            if not fname.lower().endswith(('.jpg', '.png', '.jpeg')): continue

            img_path = os.path.join(class_dir, fname)
            label_path = os.path.join(label_dir, os.path.splitext(fname)[0] + '.txt')

            # Open image to get dimensions
            with Image.open(img_path) as img:
                w, h = img.size

            # Assume object occupies entire image (centered)
            cx = 0.5
            cy = 0.5
            ww = 1.0
            hh = 1.0

            class_id = class_to_id[class_name]

            with open(label_path, 'w') as f:
                f.write(f"{class_id} {cx} {cy} {ww} {hh}\n")

print("✅ Labels auto-generated for 5 classes only!")


✅ Labels auto-generated for 5 classes only!


In [7]:
import os
import shutil

source_root = "/content/Dataset"
dest_root = "/content/flattened"

splits = ["train", "test", "validation"]

# Allowed classes only
allowed_classes = ['Human body', 'Missile', 'Rifle', 'Tank', 'Helicopter']

for split in splits:
    src_path = os.path.join(source_root, split)
    dst_img_path = os.path.join(dest_root, split, "images")
    dst_lbl_path = os.path.join(dest_root, split, "labels")

    os.makedirs(dst_img_path, exist_ok=True)
    os.makedirs(dst_lbl_path, exist_ok=True)

    class_folders = os.listdir(src_path)

    for cls in class_folders:
        if cls not in allowed_classes:
            print(f"❌ Skipping {cls} (not in 5-class list)")
            continue

        cls_path = os.path.join(src_path, cls)
        if not os.path.isdir(cls_path):
            continue

        for fname in os.listdir(cls_path):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                # Image
                src_file = os.path.join(cls_path, fname)
                dst_file = os.path.join(dst_img_path, f"{cls}_{fname}")
                shutil.copy2(src_file, dst_file)

                # Label
                label_filename = os.path.splitext(fname)[0] + ".txt"
                src_label_file = os.path.join(cls_path, "labels", label_filename)
                if os.path.exists(src_label_file):
                    dst_label_file = os.path.join(dst_lbl_path, f"{cls}_{label_filename}")
                    shutil.copy2(src_label_file, dst_label_file)

print("✅ Dataset flattened for 5 clean classes only.")


✅ Dataset flattened for 5 clean classes only.


In [8]:
# Create the updated data.yaml content
data_yaml_content = """
train: /content/flattened/train/images
val: /content/flattened/validation/images
test: /content/flattened/test/images

nc: 5
names: ['Human body', 'Missile', 'Rifle', 'Tank', 'Helicopter']
"""

# Write to file
data_yaml_path = "/content/data.yaml"
with open(data_yaml_path, "w") as f:
    f.write(data_yaml_content.strip())

data_yaml_path


'/content/data.yaml'

In [9]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [10]:
import os

# Base dataset path
base_path = "/content/flattened"
splits = ["train", "validation", "test"]
class_count = 5  # ✅ Now only 5 classes

bad_files = []

for split in splits:
    image_dir = os.path.join(base_path, split, "images")
    label_dir = os.path.join(base_path, split, "labels")

    for img_file in os.listdir(image_dir):
        if not img_file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_path = os.path.join(label_dir, label_file)

        if not os.path.exists(label_path):
            bad_files.append(f"[MISSING] {split}/labels/{label_file}")
            continue

        with open(label_path, "r") as f:
            lines = f.readlines()
            if len(lines) == 0:
                bad_files.append(f"[EMPTY] {split}/labels/{label_file}")
                continue

            for line in lines:
                parts = line.strip().split()
                if len(parts) != 5:
                    bad_files.append(f"[BAD FORMAT] {split}/labels/{label_file} → {line.strip()}")
                    break

                try:
                    class_id = int(parts[0])
                    coords = list(map(float, parts[1:]))

                    if not (0 <= class_id < class_count):
                        bad_files.append(f"[CLASS OUT OF RANGE] {split}/labels/{label_file} → class_id={class_id}")

                    if any(not (0.0 <= x <= 1.0) for x in coords):
                        bad_files.append(f"[COORD OUT OF BOUNDS] {split}/labels/{label_file} → {coords}")
                except Exception as e:
                    bad_files.append(f"[PARSE ERROR] {split}/labels/{label_file} → {line.strip()} :: {str(e)}")
                    break

print(f"\n🧹 Verification Complete. Total issues found: {len(bad_files)}\n")

for b in bad_files[:20]:  # Show only first 20 for brevity
    print("❌", b)



🧹 Verification Complete. Total issues found: 0



In [ ]:
import os
import time
import pandas as pd
from ultralytics import YOLO

# === SETUP ===
data_yaml = "data.yaml"  # Make sure this contains your 5-class dataset
model_path = "yolov8n.pt"  # Change to yolov8s/m/l if you have better GPU
output_dir = "runs/detect/fast-yolo-run"
model_save_path = os.path.join(output_dir, "weights", "best.pt")
csv_log_path = "training_metrics.csv"

# === TRAIN MODEL ===
model = YOLO(model_path)
start_time = time.time()

results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    name="fast-yolo-run",
    project="runs/detect",
    save=True,
    exist_ok=True,
    verbose=True
)

end_time = time.time()

# === LOGGING METRICS ===
# Attempt to get metrics safely
metrics = getattr(results, 'metrics', {})

log_data = {
    "epochs": getattr(results, 'epoch', 99) + 1,
    "train/box_loss": metrics.get("train/box_loss"),
    "train/cls_loss": metrics.get("train/cls_loss"),
    "train/dfl_loss": metrics.get("train/dfl_loss"),
    "val/box_loss": metrics.get("val/box_loss"),
    "val/cls_loss": metrics.get("val/cls_loss"),
    "val/dfl_loss": metrics.get("val/dfl_loss"),
    "precision": metrics.get("metrics/precision(B)"),
    "recall": metrics.get("metrics/recall(B)"),
    "mAP50": metrics.get("metrics/mAP50(B)"),
    "mAP50-95": metrics.get("metrics/mAP50-95(B)"),
    "train_time(min)": round((end_time - start_time) / 60, 2),
    "model_path": model_save_path
}

# === SAVE TO CSV ===
df = pd.DataFrame([log_data])
df.to_csv(csv_log_path, index=False)

# === DONE ===
print("✅ Training complete.")
print(f"📦 Model saved to: {model_save_path}")
print(f"📊 Metrics saved to: {csv_log_path}")


Ultralytics 8.3.156 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=fast-yolo-run, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretrained=True, prof

train: Scanning /content/flattened/train/labels.cache... 1000 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1000/1000 [00:00<?, ?it/s]

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1008.9±888.0 MB/s, size: 220.7 KB)


val: Scanning /content/flattened/validation/labels.cache... 469 images, 0 backgrounds, 0 corrupt: 100%|██████████| 469/469 [00:00<?, ?it/s]


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     96/100      3.05G    0.04336    0.09699     0.8493          8        640: 100%|██████████| 63/63 [00:20<00:00,  3.08it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:06<00:00,  2.31it/s]

                   all        469        469      0.881      0.808      0.923       0.92



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


     97/100      3.05G    0.03981    0.08382     0.8519         16        640:  19%|█▉        | 12/63 [00:03<00:16,  3.07it/s]